# Weekly Workflow — BBO Capstone

Slim notebook for each week: load data, fit surrogates, generate queries, format for portal.
Run cells in order. After receiving results, use **data_management.ipynb** to load and update, then return here for the next week.

## 1. Setup and load data

**Week 6 approach** (after Week 5 post-mortem: only F3 improved; F5/F6/F7 regressed):

- **F1**: Hard-code [0.5, 0.5] — no signal after 15 samples; space-filling, center never queried.
- **F2**: **GP** with noise=0.05, EI ξ=0.005, focus_radius=0.04 near W4 best [0.70, 0.93] (SVR failed in W5).
- **F3**: GP-ARD, EI ξ=0.001, focus_radius=0.03 near W5 best [0.384, 0.390, 0.467].
- **F4**: GP-ARD, EI ξ=0.0001, focus_radius=0.015 near W2 best [0.417, 0.409, 0.355, 0.427].
- **F5**: Hard-code [0.999999, 0.999999, 0.999999, 0.999999] — unimodal, push upper boundary (surrogate escaped in W5).
- **F6**: **GP-ARD** (not MLP), EI ξ=0.005, focus_radius=0.03 near W1 best [0.248, 0.315, 0.412, 0.758, 0.000].
- **F7**: **GP-ARD** (not MLP), EI ξ=0.005, focus_radius=0.04 near W3 best [0.070, 0.246, 0.464, 0.194, 0.366, 0.770].
- **F8**: **SVR** + EI ξ=0.01, focus_radius=0.02 near W2 best [0.115, 0.174, 0.100, 0.100, 0.895, 0.308, 0.100, 0.538].

Run the cells below to load functions and historical results, then edit the strategy dict in **Section 2** if needed and generate queries.

In [ ]:
import sys
from pathlib import Path
# Allow imports from src when run from notebooks/ or from project root
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))

import numpy as np
from src.data import FunctionData, DATA_DIR, initialize_all_weeks
from src.surrogates import GPSurrogate, SVMSurrogate, MLPSurrogate
from src.acquisition import optimize_acquisition_enhanced, optimize_acquisition_with_regional_focus
from src.utils import format_for_portal, plot_progress, display_competition_summary

np.random.seed(42)
print("✓ Imports ready")
# If you get ImportError for SVMSurrogate/MLPSurrogate: use Kernel → Restart, then run again (clears cached bytecode).

✓ Imports ready


In [2]:
# Load all 8 functions (from data/function_1 .. function_8)
functions = {i: FunctionData(i, data_dir=DATA_DIR) for i in range(1, 9)}
for i, f in functions.items():
    print(f"  {f}")
print("✓ Functions loaded")

  Function 1: 2D, 10 samples, best=0.000000
  Function 2: 2D, 10 samples, best=0.611205
  Function 3: 3D, 15 samples, best=-0.034835
  Function 4: 4D, 30 samples, best=-4.025542
  Function 5: 4D, 20 samples, best=1088.859618
  Function 6: 5D, 20 samples, best=-0.714265
  Function 7: 6D, 30 samples, best=1.364968
  Function 8: 8D, 40 samples, best=9.598482
✓ Functions loaded


In [ ]:
# Initialize with all historical weeks (from data/results/week_1, week_2, ...)
num_weeks = initialize_all_weeks(functions)
print(f"✓ Initialized with {num_weeks} weeks of results")

## 2. Week-specific strategies

Edit the strategy per function below. Then run the next cell to generate queries.

In [3]:
# Week 6 strategies (post Week 5: F1/F5 manual; F2/F3/F4/F6/F7 GP or GP-ARD; F8 SVR)
# Use "manual_query": np.array([...]) to bypass surrogate and use fixed point.
# Use "use_ard": True with surrogate "gp" for GP-ARD.
CURRENT_WEEK = 6

# Reference points (best known per function after Week 5)
f1_center = np.array([0.5, 0.5])
f2_w4_best = np.array([0.700, 0.934])
f3_w5_best = np.array([0.384, 0.390, 0.467])  # W5 new best
f4_w2_best = np.array([0.417, 0.409, 0.355, 0.427])
f5_upper = np.array([0.999999, 0.999999, 0.999999, 0.999999])  # push boundary (max valid)
f6_w1_best = np.array([0.248, 0.315, 0.412, 0.758, 0.0])
f7_w3_best = np.array([0.070, 0.246, 0.464, 0.194, 0.366, 0.770])
f8_w2_best = np.array([0.115, 0.174, 0.100, 0.100, 0.895, 0.308, 0.100, 0.538])

strategies = {
    1: {"manual_query": f1_center},
    2: {"surrogate": "gp", "gp_noise": 0.05, "acq_func": "ei", "xi": 0.005, "use_regional_focus": True, "focus_region": f2_w4_best, "focus_radius": 0.04, "n_random": 1500},
    3: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 0.001, "use_regional_focus": True, "focus_region": f3_w5_best, "focus_radius": 0.03, "n_random": 1500},
    4: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 0.0001, "use_regional_focus": True, "focus_region": f4_w2_best, "focus_radius": 0.015, "expand_search": False, "n_random": 1500},
    5: {"manual_query": f5_upper},
    6: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 0.005, "use_regional_focus": True, "focus_region": f6_w1_best, "focus_radius": 0.03, "n_random": 1500},
    7: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 0.005, "use_regional_focus": True, "focus_region": f7_w3_best, "focus_radius": 0.04, "n_random": 1500},
    8: {"surrogate": "svr", "acq_func": "ei", "xi": 0.01, "use_regional_focus": True, "focus_region": f8_w2_best, "focus_radius": 0.02, "n_random": 1200},
}
print(f"✓ Strategies set for Week {CURRENT_WEEK}")

✓ Strategies set for Week 5


## 3. Generate queries

In [4]:
def _make_surrogate(surrogate_key: str, func_data, strategy: dict):
    key = surrogate_key.lower()
    if key == "gp":
        use_ard = strategy.get("use_ard", False)
        noise = strategy.get("gp_noise", 1e-5)
        return GPSurrogate(length_scale=0.5, optimize=True, noise=noise, use_ard=use_ard)
    if key == "svr":
        return SVMSurrogate(C=10.0, epsilon=0.1, n_bootstrap=20)
    if key == "mlp":
        return MLPSurrogate(hidden_sizes=(64, 32), dropout=0.1, n_mc_samples=50)
    return GPSurrogate(length_scale=0.5, optimize=True)

queries = {}
surrogates = {}

for func_id in range(1, 9):
    func_data = functions[func_id]
    s = strategies.get(func_id, {"acq_func": "ucb", "beta": 2.0})
    if "manual_query" in s:
        x = np.asarray(s["manual_query"], dtype=float)
        queries[func_id] = x
        surrogates[func_id] = None
        print(f"  F{func_id}: MANUAL → x = {np.array2string(x, precision=4)}")
        continue
    surrogate_key = s.get("surrogate", "gp")
    surrogate = _make_surrogate(surrogate_key, func_data, s)
    surrogate.fit(func_data.inputs, func_data.outputs)
    acq_func = s.get("acq_func", "ucb")
    acq_params = {k: v for k, v in s.items() if k in ("beta", "xi")}
    if s.get("use_regional_focus") and s.get("focus_region") is not None:
        x, mu, sigma = optimize_acquisition_with_regional_focus(
            surrogate, func_data, acq_func=acq_func,
            focus_region=s["focus_region"], focus_radius=s.get("focus_radius", 0.15),
            n_random=s.get("n_random", 1000), expand_search=s.get("expand_search", True),
            **acq_params
        )
    else:
        x, mu, sigma = optimize_acquisition_enhanced(
            surrogate, func_data, acq_func=acq_func,
            n_random=s.get("n_random", 1000), **acq_params
        )
    queries[func_id] = x
    surrogates[func_id] = surrogate
    print(f"  F{func_id}: {acq_func.upper()} → x = {np.array2string(x, precision=4)}")

print("✓ Queries generated")

/home/robin/Personal_Development/Capstone-Project-ML-AI-Imperial-College/venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.01. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  F1: UCB → x = [0.3415 0.76  ]
  F2: EI → x = [0.7875 0.98  ]
  F3: EI → x = [0.3843 0.3899 0.4674]
  F4: EI → x = [0.4311 0.4118 0.3711 0.4149]
  F5: EI → x = [0.2936 0.8403 0.9734 0.9146]
  F6: UCB → x = [0.2167 0.2224 0.4042 0.8806 0.02  ]
  F7: UCB → x = [0.0264 0.4633 0.2743 0.0701 0.3059 0.8442]
  F8: EI → x = [0.1013 0.1521 0.0615 0.1278 0.9169 0.2997 0.0393 0.5522]
✓ Queries generated


## 4. Format and submit

In [5]:
format_for_portal(queries, title=f"WEEK {CURRENT_WEEK} QUERIES — READY FOR SUBMISSION")

╔==============================================================================╗
║                    WEEK 5 QUERIES — READY FOR SUBMISSION                     ║
╠==============================================================================╣
║ Function 1: 0.341514-0.759990                                                ║
║ Function 2: 0.787540-0.980000                                                ║
║ Function 3: 0.384322-0.389940-0.467379                                       ║
║ Function 4: 0.431150-0.411799-0.371084-0.414871                              ║
║ Function 5: 0.293633-0.840346-0.973393-0.914579                              ║
║ Function 6: 0.216747-0.222433-0.404206-0.880582-0.020000                     ║
║ Function 7: 0.026382-0.463272-0.274251-0.070086-0.305867-0.844177            ║
║ Function 8: 0.101309-0.152052-0.061516-0.127773-0.916922-0.299651-0.039272-0.552163 ║
╚==============================================================================╝

✓ Copy the formatted

## 5. Optional: view progress

In [ ]:
display_competition_summary(functions)
plot_progress(functions)